In [ ]:
# ============================================
# Parkinson's Disease vs Control EEG Classifier
# Full Pipeline: BIDS loading, Preprocessing, ICA, Feature Extraction, ML
# Dataset: ds002778 (BIDS format)
# ============================================

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
import mne
from mne_bids import BIDSPath, read_raw_bids
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import openneuro

In [ ]:
import os
import openneuro

# --------------------------------------------
# STEP 1: Download and Set Up Multiple Datasets
# --------------------------------------------

datasets = {
    "PD1": "ds002778",
    "PD2": "ds003490",
    "PD3": "ds003506"
}

for folder_name, dataset_id in datasets.items():
    os.makedirs(folder_name, exist_ok=True)
    print(f"⬇️ Downloading {dataset_id} into {folder_name} ...")
    openneuro.download(dataset=dataset_id, target_dir=folder_name)
    print(f"✅ Download complete for {folder_name}")

# --------------------------------------------
# STEP 2: Collect All Subjects Across Datasets
# --------------------------------------------

all_subject_paths = []

for dataset_root in datasets.keys():
    dataset_path = dataset_root
    subjects = [d for d in os.listdir(dataset_path) if d.startswith("sub-")]
    
    for sub in subjects:
        # Add all subject/session pairs to processing list
        subject_path = os.path.join(dataset_path, sub)
        all_subject_paths.append((dataset_path, sub))

print(f"👥 Total Subjects Found: {len(all_subject_paths)}")
for root, subject in all_subject_paths:
    print(f" - {subject} (from {root})")

In [ ]:
import os
import pandas as pd
import torch
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Define all BIDS roots (already downloaded)
bids_roots = ["PD1", "PD2", "PD3"]

features = []
labels = []
subject_ids = []

for bids_root in bids_roots:
    participants_file = os.path.join(bids_root, "participants.tsv")

    if not os.path.exists(participants_file):
        print(f"⚠️ Missing {participants_file}, skipping {bids_root}")
        continue

    df = pd.read_csv(participants_file, sep='\t')

    # Handle missing columns
    for col in ['gender', 'hand', 'MMSE', 'NAART', 'disease_duration']:
        if col not in df.columns:
            df[col] = np.nan

    le_gender = LabelEncoder()
    le_hand = LabelEncoder()

    df['gender'] = le_gender.fit_transform(df['gender'].fillna('unknown'))
    df['hand'] = le_hand.fit_transform(df['hand'].fillna('unknown'))

    for subject in os.listdir(bids_root):
        subj_path = os.path.join(bids_root, subject)
        if not os.path.isdir(subj_path) or not subject.startswith("sub-"):
            continue

        subj_row = df[df["participant_id"] == subject]
        if subj_row.empty:
            print(f"Skipping {subject} in {bids_root}: not in participants.tsv")
            continue
        subj_row = subj_row.iloc[0]

        age = subj_row.get("age", np.nan)
        gender = subj_row.get("gender", -1)
        hand = subj_row.get("hand", -1)
        mmse = subj_row.get("MMSE", 0)
        naart = subj_row.get("NAART", 0)
        disease_duration = subj_row.get("disease_duration", 0)

        for session in os.listdir(subj_path):
            if not session.startswith("ses-"):
                continue

            session_path = os.path.join(subj_path, session)
            if not os.path.isdir(session_path):
                continue

            # Dataset-specific session label mapping
            if bids_root == "PD1":
                # Original labeling logic
                if session == "ses-hc":
                    label = 0  # Control
                elif session == "ses-on":
                    label = 1  # PD-On
                elif session == "ses-off":
                    label = 2  # PD-Off
                else:
                    print(f"Unknown session {session} for {subject} in {bids_root}, skipping.")
                    continue

            elif bids_root in ["PD2", "PD3"]:
                # PD2 and PD3 usually have ses-01, ses-02 — assign labels accordingly:
                # **You must adjust this mapping based on dataset docs!**
                session_label_map = {
                    "ses-01": 0,  # e.g., Control
                    "ses-02": 1   # e.g., PD patient
                }
                if session in session_label_map:
                    label = session_label_map[session]
                else:
                    print(f"Unknown session {session} for {subject} in {bids_root}, skipping.")
                    continue
            else:
                print(f"Unknown dataset root {bids_root}, skipping session.")
                continue

            feature_vector = np.array([
                age if not pd.isna(age) else 0,
                gender,
                hand,
                mmse if not pd.isna(mmse) else 0,
                naart if not pd.isna(naart) else 0,
                disease_duration if not pd.isna(disease_duration) else 0
            ])
            features.append(feature_vector)
            labels.append(label)
            subject_ids.append(f"{bids_root}_{subject}_{session}")

# Convert features list to numpy array first to avoid warning
features_array = np.array(features)
features_tensor = torch.tensor(features_array, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

print(f"Features tensor shape: {features_tensor.shape}")
print(f"Labels tensor shape: {labels_tensor.shape}")
print(f"Label counts: {np.bincount(labels_tensor.numpy())}")


## Step 1: Define Feature Extraction Function

In [ ]:
import mne
from mne.preprocessing import ICA
import numpy as np
from scipy.stats import skew
from sklearn.decomposition import PCA

def extract_band_power_epochs(
    epochs,
    bands=[(1, 4), (4, 8), (8, 13), (13, 30)],
    compress=False,
    apply_pca=True,
    pca_components=50
):
    """
    Extract band power features from ICA-cleaned EEG epochs (Parkinson's Dataset).

    Parameters:
    - epochs: Cleaned and filtered mne.Epochs object
    - bands: Frequency bands [(fmin, fmax)]
    - compress: If True, aggregate features across channels (mean, std, skew)
    - apply_pca: If True, apply PCA for dimensionality reduction
    - pca_components: Number of PCA components to retain

    Returns:
    - X: Feature matrix [n_epochs, n_features]
    - feature_names: List of feature names (or PCA components)
    - pca_model: Fitted PCA object or None
    """
    
    # Ensure only EEG channels are used
    epochs.pick_types(eeg=True)

    # Remove bad channels if marked
    epochs.drop_channels([ch for ch in epochs.info['bads'] if ch in epochs.ch_names])

    # Compute PSD (power spectral density)
    psd = epochs.compute_psd(method='welch', fmin=1., fmax=40., n_fft=256)
    psd_data, freqs = psd.get_data(return_freqs=True)
    ch_names = epochs.info['ch_names']

    X = []

    if compress:
        # Aggregated band power stats
        feature_names = []
        for fmin, fmax in bands:
            label = f"{fmin}-{fmax}Hz"
            feature_names.extend([f"{label}_mean", f"{label}_std", f"{label}_skew"])

        for trial in psd_data:
            features = []
            for fmin, fmax in bands:
                idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                band_power = trial[:, idx].mean(axis=1)
                features.extend([
                    np.mean(band_power),
                    np.std(band_power),
                    skew(band_power)
                ])
            X.append(features)
    
    else:
        # Raw channel-band features
        feature_names = []
        for fmin, fmax in bands:
            for ch in ch_names:
                feature_names.append(f"{fmin}-{fmax}Hz_{ch}")

        for trial in psd_data:
            features = []
            for fmin, fmax in bands:
                idx = np.logical_and(freqs >= fmin, freqs <= fmax)
                band_power = trial[:, idx].mean(axis=1)
                features.extend(band_power)
            X.append(features)

    X = np.array(X)
    pca_model = None

    if apply_pca:
        n_samples, n_features = X.shape
        if pca_components is None or pca_components > min(n_samples, n_features):
            pca_components = min(n_samples, n_features)
            print(f"Adjusted PCA to {pca_components} components")

        print(f"Applying PCA: {X.shape[1]} → {pca_components}")
        pca_model = PCA(n_components=pca_components, random_state=42)
        X = pca_model.fit_transform(X)
        feature_names = [f"PCA_{i+1}" for i in range(X.shape[1])]

    return X, feature_names, pca_model



## Step 2: Preprocess and Extract for Subject

In [ ]:
import os
import time
import numpy as np
import mne
from mne.preprocessing import ICA, create_eog_epochs
from mne_bids import BIDSPath, read_raw_bids

# If not already defined, import your feature extraction function
# from features import extract_band_power_epochs

# --- Custom session-to-label mapper ---
def categorize_session(session_name):
    if 'on' in session_name:
        return 'pd-on'
    elif 'off' in session_name:
        return 'pd-off'
    elif 'hc' in session_name:
        return 'ctl'
    else:
        return 'unknown'

# --- Main Subject Processing Function ---
def process_subject(subject, session, bids_root, duration=2.0, overlap=1.0):
    try:
        print(f"🚀 Processing Subject: {subject}, Session: {session}")
        
        label = categorize_session(session)
        if label == 'unknown':
            print(f"❌ Unknown session type for {session}")
            return [], [], []

        bids_path = BIDSPath(
            root=bids_root,
            subject=subject.replace('sub-', ''),
            session=session.replace('ses-', ''),
            task='Rest',
            datatype='eeg',
            suffix='eeg',
            extension='.set'
        )
        print(f"📂 Loading BIDS: {bids_path}")

        raw = read_raw_bids(bids_path=bids_path, verbose=False)

        # Set channel types (adjust EXG channel names if needed)
        eog_channels = [ch for ch in raw.ch_names if 'EXG' in ch]
        for ch in eog_channels:
            raw.set_channel_types({ch: 'eog'})

        raw.set_montage("standard_1020", on_missing="ignore")
        raw.load_data()

        # --- Filtering ---
        raw.filter(1., 40.)
        raw.notch_filter(60.)
        raw.set_eeg_reference('average')

        # --- ICA ---
        print("🧹 Running ICA...")
        ica = ICA(n_components=20, random_state=42)
        ica.fit(raw)
        eog_inds, _ = ica.find_bads_eog(raw)
        ica.exclude = eog_inds
        raw = ica.apply(raw)

        # --- Epoching ---
        print("🔪 Epoching...")
        epochs = mne.make_fixed_length_epochs(raw, duration=duration, overlap=overlap, preload=True)
        if len(epochs) == 0:
            print(f"⚠️ No epochs found for {subject} {session}")
            return [], [], []

        # --- Feature Extraction ---
        print("🧠 Extracting features...")
        X_subject, feature_names = extract_band_power_epochs(
            epochs,
            compress=False,
            apply_pca=False
        )
        y_subject = [label] * len(X_subject)
        print(f"✅ Extracted {len(X_subject)} samples")

        return X_subject, y_subject, feature_names

    except Exception as e:
        print(f"❌ Error in {subject} {session}: {e}")
        return [], [], []

# --- Optional Entity Fetcher ---
def get_entity_vals(bids_root, entity_key='subject'):
    """
    Returns sorted unique subject/session identifiers from the dataset
    """
    vals = []
    for root, dirs, files in os.walk(bids_root):
        for name in dirs:
            if entity_key == 'subject' and name.startswith('sub-'):
                vals.append(name)
            elif entity_key == 'session' and name.startswith('ses-'):
                vals.append(name)
    return sorted(list(set(vals)))


## Step 3: Loop Over All Subjects

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import LabelEncoder

# Maps dataset name to (root folder, participant file)
datasets = {
    "PD1": ("PD1", "participants.tsv"),
    "PD2": ("PD2", "participants50.tsv"),
    "PD3": ("PD3", "participants56.tsv"),
}

# Prepare accumulators
features = []
labels = []
subject_ids = []

# Loop over all datasets
for dataset_name, (bids_root, participant_file) in datasets.items():
    print(f"\n📁 Processing dataset: {dataset_name}")
    
    # Load participant-level metadata
    df = pd.read_csv(os.path.join(bids_root, participant_file), sep='\t')
    
    # Encode categorical variables
    le_gender = LabelEncoder()
    df['gender'] = le_gender.fit_transform(df['gender'])

    le_hand = LabelEncoder()
    df['hand'] = le_hand.fit_transform(df['hand'])

    # Walk through all subject/session folders
    for subject in os.listdir(bids_root):
        subj_path = os.path.join(bids_root, subject)
        if not os.path.isdir(subj_path) or not subject.startswith("sub-"):
            continue

        subj_row = df[df["participant_id"] == subject]
        if subj_row.empty:
            print(f"❌ Skipping {subject}: not in {participant_file}")
            continue
        subj_row = subj_row.iloc[0]

        # Extract demographic/clinical data
        age = subj_row.get("age", 0)
        gender = subj_row.get("gender", 0)
        hand = subj_row.get("hand", 0)
        mmse = subj_row.get("MMSE", 0)
        naart = subj_row.get("NAART", 0)
        disease_duration = subj_row.get("disease_duration", 0)
        disease_duration = disease_duration if not pd.isna(disease_duration) else 0

        # Loop through available sessions
        for session in os.listdir(subj_path):
            if not session.startswith("ses-"):
                continue

            session_path = os.path.join(subj_path, session)
            if not os.path.isdir(session_path):
                continue

            # Label logic per dataset
            if dataset_name == "PD1":
                if session == "ses-hc":
                    label = 0
                elif session == "ses-on":
                    label = 1
                elif session == "ses-off":
                    label = 2
                else:
                    print(f"⚠️ Unknown session {session} in {subject}, skipping")
                    continue
            else:  # PD2 or PD3
                if session == "ses-01":
                    # HC if disease_duration is NaN or 0
                    label = 0 if pd.isna(subj_row["disease_duration"]) or subj_row["disease_duration"] == 0 else 1
                elif session == "ses-02":
                    label = 2
                else:
                    print(f"⚠️ Unknown session {session} in {subject}, skipping")
                    continue

            # Add feature vector and label
            features.append(np.array([age, gender, hand, mmse, naart, disease_duration]))
            labels.append(label)
            subject_ids.append(f"{subject}_{session}")

# Convert to tensors
features_tensor = torch.tensor(features, dtype=torch.float32)
labels_tensor = torch.tensor(labels, dtype=torch.long)

# Print summary
print("\n✅ Final Summary:")
print(f"Feature tensor shape: {features_tensor.shape}")
print(f"Label tensor shape: {labels_tensor.shape}")
print(f"Label distribution: {np.bincount(labels_tensor.numpy())}")


### Debug

## Step 4: Train ML Classifier

### Train Random Forest Classifier

In [ ]:
from sklearn.model_selection import train_test_split

# Assuming X and y have been processed and are available

# Split data into training and test sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Print the shapes of the resulting datasets
print(f"✅ Shape of X_train: {X_train.shape}")
print(f"✅ Shape of X_test: {X_test.shape}")
print(f"✅ Shape of y_train: {y_train.shape}")
print(f"✅ Shape of y_test: {y_test.shape}")

# Check the first epoch in the dataset
print(f"First sample (epoch) shape: {X[0].shape}")
print(f"First sample (epoch) data: {X[0]}")



In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Random Forest Model
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


# ***Decision Tree***

In [ ]:
!pip install plotly


In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Decision Tree Model
    model = DecisionTreeClassifier(
        max_depth=20,
        class_weight='balanced',
        random_state=42
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")



In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

# === Plot the Final Decision Tree Model with Adjustments ===
plt.figure(figsize=(150, 75))  # Decrease the size of the figure to avoid overcrowding

# Visualizing the decision tree using plot_tree function
plot_tree(
    model,  # The trained decision tree model
    filled=True,  # Color the nodes to make them more readable
    feature_names=['Feature' + str(i) for i in range(X_train_scaled.shape[1])],  # Feature names (e.g., Feature0, Feature1, ...)
    class_names=class_names,  # Class names for each target class (Control, PD-On, PD-Off)
    rounded=True,  # Round the corners of the nodes for better aesthetics
    fontsize=10,  # Reduce font size to avoid overcrowding
    proportion=True,  # Show proportion of samples in each node
    precision=2  # Precision of float values displayed on the tree
)

# Add a title to the plot
plt.title("Final Decision Tree Visualization", fontsize=8)

# Adjust layout to avoid clipping of labels and titles
plt.subplots_adjust(left=0.05, right=0.95, top=0.9, bottom=0.05)

# Display the plot
plt.show()



### Step 4.2 : Train Support Vector Machine Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 SVM Model (with probability estimation for multi-class)
    model = SVC(
        kernel='linear',  # Using linear kernel for simplicity, can be adjusted
        C=1,  # Regularization parameter
        class_weight='balanced',  # Handle class imbalance
        random_state=42,
        probability=True,  # Enable probability estimates
        verbose=False  # Disable verbose output
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


## Step 4.3: Train Logistic Regression Classifier

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_fscore_support

# Assuming X and y have been processed and are available
# Ensure y is a 1D NumPy array if it’s a pandas series or dataframe
if isinstance(y, pd.Series):
    y = y.values
elif isinstance(y, pd.DataFrame):
    y = y.squeeze().values

# 🎯 Class names
class_names = ['Control', 'PD-On', 'PD-Off']
n_classes = len(class_names)

# ⚙️ Cross-Validation Setup
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 🧪 Metrics tracking
all_metrics = {
    "accuracy": [],
    "precision": [],
    "recall": [],
    "f1": [],
    "auc": []
}

# 🔄 Fold Loop
for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n📁 Fold {fold}")
    
    # Splitting data into training and test sets
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    # ⚖️ Preprocessing
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # 🌳 Logistic Regression Model
    model = LogisticRegression(
        max_iter=1000,  # Ensure convergence
        class_weight='balanced',  # Handle class imbalance
        random_state=42,
        multi_class='ovr',  # One-vs-rest for multi-class classification
        solver='liblinear'  # Using liblinear solver for small datasets
    )
    model.fit(X_train_scaled, y_train)
    
    # Make predictions and probabilities
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)
    
    # 📈 Metrics
    acc = np.mean(y_pred == y_test)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro')
    y_test_bin = label_binarize(y_test, classes=list(range(n_classes)))  # Binarize for multi-class ROC
    auc_score = roc_auc_score(y_test_bin, y_proba, average='macro', multi_class='ovr')

    # 📊 Append
    all_metrics["accuracy"].append(acc)
    all_metrics["precision"].append(prec)
    all_metrics["recall"].append(rec)
    all_metrics["f1"].append(f1)
    all_metrics["auc"].append(auc_score)

    # 🧾 Classification Report
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    
    # Display the classification report
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # 🔵 Confusion Matrix (blue)
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix - Fold {fold}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # 📉 ROC Curve (macro-averaged)
    fpr = dict()
    tpr = dict()
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_test_bin[:, i], y_proba[:, i])
    
    # Compute macro-average
    all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))
    mean_tpr = np.zeros_like(all_fpr)
    for i in range(n_classes):
        mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])
    mean_tpr /= n_classes
    
    plt.figure(figsize=(6, 5))
    plt.plot(all_fpr, mean_tpr, color='blue', label=f'Macro-Averaged ROC (AUC = {auc_score:.2f})')
    plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Chance')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'Macro ROC Curve - Fold {fold}')
    plt.legend(loc='lower right')
    plt.tight_layout()
    plt.show()

# 📊 Overall Summary
print("\n📊 Overall Cross-Validation Summary:")
for key in all_metrics:
    mean_val = np.mean(all_metrics[key])
    print(f"Mean {key.capitalize():<9}: {mean_val:.4f}")


## **Transformer Model**

In [ ]:
!pip install torch torchvision

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

**BLOCK 1: Preprocessing and DataLoader**

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import torch
from torch.utils.data import DataLoader, TensorDataset
import numpy as np

# Entire dataset (X and y)
X = np.concatenate([X_train, X_test], axis=0)  # shape (4497, 160)
y = np.concatenate([y_train, y_test], axis=0)  # shape (4497,)

X = X.astype(np.float32)
y = y.astype(np.int64)

# K-Fold cross-validation will split this
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Store all folds' loaders, weights, scalers
fold_data = []

for fold_idx, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"📂 Preparing Fold {fold_idx}...")

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Standardize using only training set
    scaler = StandardScaler()
    X_train_std = scaler.fit_transform(X_train)
    X_val_std = scaler.transform(X_val)

    # Convert to PyTorch tensors
    X_train_tensor = torch.tensor(X_train_std, dtype=torch.float32)
    X_val_tensor = torch.tensor(X_val_std, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype=torch.long)
    y_val_tensor = torch.tensor(y_val, dtype=torch.long)

    # Compute class weights for this fold
    class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    # Create DataLoaders
    batch_size = 64
    train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_val_tensor, y_val_tensor), batch_size=batch_size)

    fold_data.append({
        'train_loader': train_loader,
        'val_loader': val_loader,
        'class_weights': class_weights_tensor,
        'scaler': scaler,
        'X_val': X_val_std,
        'y_val': y_val,
    })



**BLOCK 2: Transformer Model Definition**

In [ ]:
import torch
import torch.nn as nn

class EEGTransformer(nn.Module):
    def __init__(self, input_dim=160, num_classes=3, embed_dim=128, num_heads=4, num_layers=2, dropout=0.3):
        super().__init__()
        
        self.embedding = nn.Linear(input_dim, embed_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, 
            nhead=num_heads, 
            dropout=dropout,
            batch_first=True
        )
        
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        # x shape: (B, input_dim)
        x = self.embedding(x).unsqueeze(1)  # (B, 1, embed_dim)
        x = self.transformer_encoder(x)     # (B, 1, embed_dim)
        x = x.squeeze(1)                    # (B, embed_dim)
        return self.classifier(x)           # (B, num_classes)


**BLOCK 3: Training Setup**

In [ ]:
from transformers import get_cosine_schedule_with_warmup
import torch.optim as optim
import torch.nn as nn

def initialize_training_components(input_dim, num_classes, class_weights_tensor, train_loader, epochs=40, lr=5e-4):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Initialize model
    model = EEGTransformer(input_dim=input_dim, num_classes=num_classes).to(device)

    # Loss function with class weights
    loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=lr)

    # Cosine LR scheduler with warmup
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),  # 10% warmup by default
        num_training_steps=total_steps
    )

    return model, loss_fn, optimizer, scheduler, device


**BLOCK 4: Training and Evaluation Loop with tqdm**

In [ ]:
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import torch
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

def train_one_epoch(model, loader, optimizer, loss_fn, scheduler, device):
    model.train()
    total_loss = 0
    correct, total = 0, 0
    loop = tqdm(loader, desc="Train", leave=False)
    
    for xb, yb in loop:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        preds = model(xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        correct += (preds.argmax(1) == yb).sum().item()
        total += yb.size(0)

        loop.set_postfix(loss=loss.item(), acc=correct / total)

    avg_loss = total_loss / len(loader)
    avg_acc = correct / total
    return avg_loss, avg_acc

def evaluate(model, loader, device, return_preds=False, show_plot=True):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = logits.argmax(1).cpu().numpy()

            y_prob.extend(probs)
            y_pred.extend(preds)
            y_true.extend(yb.numpy())

    if return_preds:
        return y_true, y_pred, y_prob

    # Report and Confusion Matrix
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()


**BLOCK 5: Training Loop**

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

def plot_confusion_matrix(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])
    plt.figure(figsize=(6, 5))
    sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    plt.title("Confusion Matrix")
    plt.ylabel("True")
    plt.xlabel("Predicted")
    plt.tight_layout()
    plt.show()

def plot_macro_roc_curve(y_true, y_prob):
    y_true_bin = label_binarize(y_true, classes=[0, 1, 2])
    y_prob = np.array(y_prob)
    fpr, tpr, _ = roc_curve(y_true_bin.ravel(), y_prob.ravel())
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(6, 5))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"Macro-average ROC curve (AUC = {roc_auc:.4f})")
    plt.plot([0, 1], [0, 1], color='navy', lw=1, linestyle='--')
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("Macro-Averaged ROC Curve")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.tight_layout()
    plt.show()

# Your updated training + evaluation loop:
num_epochs = 40
all_y_true, all_y_pred, all_y_prob = [], [], []

for fold_idx, fold in enumerate(fold_data, 1):
    print(f"\n📁 ===== Fold {fold_idx} =====")

    train_loader = fold["train_loader"]
    val_loader = fold["val_loader"]
    class_weights = fold["class_weights"]

    model, loss_fn, optimizer, scheduler, device = initialize_training_components(
        input_dim=160,
        num_classes=3,
        class_weights_tensor=class_weights,
        train_loader=train_loader,
        epochs=num_epochs,
        lr=5e-4
    )

    # Train loop
    for epoch in range(1, num_epochs + 1):
        print(f"\n🌟 Epoch {epoch}/{num_epochs} — Fold {fold_idx}")
        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, loss_fn, scheduler, device)
        print(f"✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")

    # Final evaluation on validation set for this fold
    y_true, y_pred, y_prob = evaluate(model, val_loader, device, return_preds=True, show_plot=False)

    # Print classification report
    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["CTL", "PD-On", "PD-Off"], digits=4))

    # Plot confusion matrix
    plot_confusion_matrix(y_true, y_pred)

    # Plot macro-average ROC AUC curve
    plot_macro_roc_curve(y_true, y_prob)

    # Collect all fold data for combined stats later
    all_y_true.extend(y_true)
    all_y_pred.extend(y_pred)
    all_y_prob.extend(y_prob)


**BLOCK 6:Evauluate**

In [ ]:
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

# --- Final Classification Report and Confusion Matrix ---
print("\n📊 Final Cross-Validated Classification Report:")
print(classification_report(
    all_y_true, all_y_pred,
    target_names=["CTL", "PD-On", "PD-Off"],
    digits=4
))

cm = confusion_matrix(all_y_true, all_y_pred)
df_cm = pd.DataFrame(cm, index=["CTL", "PD-On", "PD-Off"], columns=["CTL", "PD-On", "PD-Off"])

plt.figure(figsize=(6, 5))
sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Confusion Matrix (All Folds Combined)", fontsize=13)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


# --- Macro-Averaged ROC Curve ---

# Binarize labels for multiclass ROC
y_true_bin = label_binarize(all_y_true, classes=[0, 1, 2])
y_prob = np.array(all_y_prob)

n_classes = y_true_bin.shape[1]
fpr = dict()
tpr = dict()
roc_auc = dict()

# Compute ROC curve and AUC for each class
for i in range(n_classes):
    fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_prob[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])

# Aggregate all FPR points
all_fpr = np.unique(np.concatenate([fpr[i] for i in range(n_classes)]))

# Interpolate all TPRs at these points and average them
mean_tpr = np.zeros_like(all_fpr)
for i in range(n_classes):
    mean_tpr += np.interp(all_fpr, fpr[i], tpr[i])

mean_tpr /= n_classes

# Compute macro-average AUC
macro_auc = auc(all_fpr, mean_tpr)

# Plot macro-average ROC
plt.figure(figsize=(7, 6))
plt.plot(all_fpr, mean_tpr, color='navy', lw=2, label=f'Macro-average ROC (AUC = {macro_auc:.4f})')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', lw=1)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('📈 Macro-Averaged ROC Curve (All Folds Combined)', fontsize=13)
plt.legend(loc='lower right')
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()

